# 05 检索系统开发第一部分 — 查询理解与增强

> **英文 query 优先**。双库 smoke test：`chroma_db`（样本）与 `chroma_db_full`（全量），对比 HNSW bin 与 query 耗时。

- **C0** 环境与路径
- **C1** 查询增强模块
- **C2** 导出增强样例
- **C3** 样本库 smoke test
- **C4** 全量库 smoke test
- **C5** 对比与导出

## C0 环境与路径

配置 Stage04/05 路径、`sys.path`，检查样本库与全量库是否存在。

In [1]:
import json
import sys
from pathlib import Path

STAGE05 = Path("..").resolve()
STAGE04 = (STAGE05.parent / "04 向量化与索引构建").resolve()
SRC05 = STAGE05 / "src"
SRC04 = STAGE04 / "src"

for p in (SRC05, SRC04):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

PERSIST_SAMPLE = STAGE04 / "data" / "chroma_db"
PERSIST_FULL = STAGE04 / "data" / "chroma_db_full"
COLLECTION_SAMPLE = "pmc_oa_comm_sample"
COLLECTION_FULL = "pmc_oa_comm_full"

print("STAGE05:", STAGE05)
print("样本库:", PERSIST_SAMPLE, PERSIST_SAMPLE.exists())
print("全量库:", PERSIST_FULL, PERSIST_FULL.exists())

STAGE05: D:\谷歌\05 检索系统开发第一部分
样本库: D:\谷歌\04 向量化与索引构建\data\chroma_db True
全量库: D:\谷歌\04 向量化与索引构建\data\chroma_db_full True


## C1 查询增强模块

加载 `MedicalQueryEnhancer`，生成英文示例查询的增强结果。

In [2]:
from query_enhancer import MedicalQueryEnhancer

enhancer = MedicalQueryEnhancer(STAGE05 / "data" / "medical_synonyms.json")

DEMO_QUERIES = [
    "What is the treatment for MI?",
    "metformin cardiovascular effects",
    "papers on malaria after 2015",
    "circadian rhythm in sliding window chunks",
]

enhanced_list = [enhancer.process(q) for q in DEMO_QUERIES]
for eq in enhanced_list:
    print("\n===", eq.original)
    print("  vector:", eq.vector_query)
    print("  keyword:", eq.keyword_query)
    print("  expanded:", eq.expanded_terms)
    print("  filters:", [(f.key, f.value, f.executable) for f in eq.filters])


=== What is the treatment for MI?
  vector: What is the treatment for MI?
  keyword: treatment mi myocardial infarction heart attack
  expanded: ['myocardial infarction', 'heart attack']
  filters: []

=== metformin cardiovascular effects
  vector: metformin cardiovascular effects
  keyword: metformin cardiovascular effects biguanide antidiabetic heart disease metformin cardiovascular effects cardiovascular outcomes metformin
  expanded: ['metformin', 'biguanide antidiabetic', 'cardiovascular', 'heart disease', 'metformin cardiovascular effects', 'cardiovascular outcomes metformin']
  filters: []

=== papers on malaria after 2015
  vector: papers on malaria after 2015
  keyword: papers malaria after 2015 plasmodium
  expanded: ['malaria', 'plasmodium']
  filters: [('year_gte', 2015, False)]

=== circadian rhythm in sliding window chunks
  vector: circadian rhythm in sliding window chunks
  keyword: circadian rhythm sliding window chunks
  expanded: []
  filters: [('strategy', 'sliding

## C2 导出增强样例

将 `EnhancedQuery` 结果写入 `outputs/samples/enhancement_examples.json`。

In [3]:
out_path = STAGE05 / "outputs" / "samples" / "enhancement_examples.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
payload = [eq.to_dict() for eq in enhanced_list]
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print("已保存:", out_path)

已保存: D:\谷歌\05 检索系统开发第一部分\outputs\samples\enhancement_examples.json


## C3 样本库 smoke test

挂载 `chroma_db`，探测 HNSW bin 并做 query 计时。

In [4]:
from embedder import DocumentEmbedder
from index_builder import ChromaIndexBuilder, count_embeddings_sqlite, repair_chroma_hnsw
from chroma_smoke import detect_hnsw_bins, timed_queries

embedder = DocumentEmbedder()
SMOKE_QUERY = enhanced_list[1].vector_query  # metformin...
WHERE = enhanced_list[3].chroma_where()  # strategy filter if any

REPAIR_HNSW = False  # 仅 query 报 hnsw 错时改 True

def attach(persist_dir, collection):
    if REPAIR_HNSW:
        repair_chroma_hnsw(persist_dir, collection)
    b = ChromaIndexBuilder(str(persist_dir), collection, embedder)
    n = count_embeddings_sqlite(persist_dir)
    h = detect_hnsw_bins(persist_dir)
    return b, n, h

builder_sample, n_sample, h_sample = attach(PERSIST_SAMPLE, COLLECTION_SAMPLE)
print("样本库条数:", n_sample)
print("样本 HNSW:", h_sample)
t_sample = timed_queries(builder_sample, SMOKE_QUERY, where_filter=WHERE, repeats=3)
print("样本 query 耗时:", t_sample)

样本库条数: 1267
样本 HNSW: {'persist_dir': 'D:\\谷歌\\04 向量化与索引构建\\data\\chroma_db', 'segments': [{'segment_id': 'c1010323-f7f1-42db-ac41-e02aa3400d7d', 'dir_exists': True, 'bins': {'data_level0.bin': True, 'link_lists.bin': True, 'length.bin': True}, 'complete_hnsw': True, 'bin_bytes': 168000}], 'any_complete_hnsw': True}
  [query] Chroma where= 失败，改用 over-fetch + Python 过滤 (InternalError)
  [query] Chroma where= 失败，改用 over-fetch + Python 过滤 (InternalError)
  [query] Chroma where= 失败，改用 over-fetch + Python 过滤 (InternalError)
  [query] Chroma where= 失败，改用 over-fetch + Python 过滤 (InternalError)
样本 query 耗时: {'query': 'metformin cardiovascular effects', 'n_results': 5, 'where_filter': {'strategy': 'sliding_window'}, 'repeats': 3, 'times_sec': [0.0131, 0.0118, 0.0114], 'mean_sec': 0.0121, 'top_ids': ['PMC523838_chunk2', 'PMC524363_chunk1', 'PMC523838_chunk4', 'PMC523838_chunk1', 'PMC523838_chunk3']}


## C4 全量库 smoke test

挂载 `chroma_db_full`，执行同一 query 的计时。

In [5]:
builder_full, n_full, h_full = attach(PERSIST_FULL, COLLECTION_FULL)
print("全量库条数:", n_full)
print("全量 HNSW:", h_full)
t_full = timed_queries(builder_full, SMOKE_QUERY, where_filter=None, repeats=3)
print("全量 query 耗时:", t_full)

全量库条数: 6107296
全量 HNSW: {'persist_dir': 'D:\\谷歌\\04 向量化与索引构建\\data\\chroma_db_full', 'segments': [{'segment_id': '5552215e-9070-40de-a5f0-521c25331367', 'dir_exists': True, 'bins': {'data_level0.bin': False, 'link_lists.bin': False, 'length.bin': False}, 'complete_hnsw': False, 'bin_bytes': 0}], 'any_complete_hnsw': False}
全量 query 耗时: {'query': 'metformin cardiovascular effects', 'n_results': 5, 'where_filter': None, 'repeats': 3, 'times_sec': [0.0174, 0.0154, 0.0157], 'mean_sec': 0.0162, 'top_ids': ['PMC12822947_chunk1', 'PMC12822947_chunk0', 'PMC12823101', 'PMC12822947_chunk2', 'PMC12823010_chunk1']}


## C5 对比与导出

汇总双库检测与耗时，导出 `chroma_smoke_compare.json`。

In [6]:
compare = {
    "smoke_query": SMOKE_QUERY,
    "sample": {
        "persist_dir": str(PERSIST_SAMPLE),
        "collection": COLLECTION_SAMPLE,
        "chunks": n_sample,
        "hnsw": h_sample,
        "timing": t_sample,
    },
    "full": {
        "persist_dir": str(PERSIST_FULL),
        "collection": COLLECTION_FULL,
        "chunks": n_full,
        "hnsw": h_full,
        "timing": t_full,
    },
    "note": "样本库与全量库条数不同，耗时仅供 HNSW/bin 状态与规模粗对比，非严格对照实验。",
}
cmp_path = STAGE05 / "outputs" / "samples" / "chroma_smoke_compare.json"
with open(cmp_path, "w", encoding="utf-8") as f:
    json.dump(compare, f, indent=2, ensure_ascii=False)
print("对比报告:", cmp_path)
print(json.dumps(compare, indent=2, ensure_ascii=False))

对比报告: D:\谷歌\05 检索系统开发第一部分\outputs\samples\chroma_smoke_compare.json
{
  "smoke_query": "metformin cardiovascular effects",
  "sample": {
    "persist_dir": "D:\\谷歌\\04 向量化与索引构建\\data\\chroma_db",
    "collection": "pmc_oa_comm_sample",
    "chunks": 1267,
    "hnsw": {
      "persist_dir": "D:\\谷歌\\04 向量化与索引构建\\data\\chroma_db",
      "segments": [
        {
          "segment_id": "c1010323-f7f1-42db-ac41-e02aa3400d7d",
          "dir_exists": true,
          "bins": {
            "data_level0.bin": true,
            "link_lists.bin": true,
            "length.bin": true
          },
          "complete_hnsw": true,
          "bin_bytes": 168000
        }
      ],
      "any_complete_hnsw": true
    },
    "timing": {
      "query": "metformin cardiovascular effects",
      "n_results": 5,
      "where_filter": {
        "strategy": "sliding_window"
      },
      "repeats": 3,
      "times_sec": [
        0.0131,
        0.0118,
        0.0114
      ],
      "mean_sec": 0.0121,
     